# Colab SSH Bootstrap

Run all cells to start an SSH tunnel into this Colab runtime.

**Order:** colab-ssh runs **before** the Google Drive mount so the `trycloudflare.com` hostname usually appears **before** the Drive consent dialog. That way browser automation (and you) can read the hostname without Drive OAuth blocking the first code cell.

Connect from Cursor after the hostname line appears:
```
scripts/connect_colab.sh <HOSTNAME>
```

The Drive cell is for a persistent clone under My Drive. If you skip Drive access, the next cell uses `/content/recsys_playground` (not persisted across sessions).

In [ ]:
!pip install colab-ssh --upgrade -q
![ -f cloudflared ] && chmod 755 cloudflared; true
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="cursorssh")

# Post hostname to ntfy.sh so the bootstrap trigger script receives it without DOM scraping
import re, time, requests as _req

_NTFY_TOPIC = "colab-ssh-allyoushawn-e3b76118-69fd-4a29-89c1-c34e07b0977e"

# Poll cloudflared log for the trycloudflare.com hostname (up to 30s)
_hostname = None
_log_candidates = [
    "/root/.colab_ssh/cloudflared.log",
    "/content/.colab_ssh/cloudflared.log",
    "./cloudflared.log",
]
for _attempt in range(10):
    for _path in _log_candidates:
        try:
            with open(_path) as _f:
                _m = re.search(r'([\w-]+\.trycloudflare\.com)', _f.read())
            if _m:
                _hostname = _m.group(1)
                break
        except FileNotFoundError:
            pass
    if _hostname:
        break
    time.sleep(3)

try:
    if _hostname:
        _req.post(f"https://ntfy.sh/{_NTFY_TOPIC}", data=_hostname.encode(), timeout=15)
        print(f"[ntfy] Hostname sent: {_hostname}")
    else:
        import subprocess as _sp
        _found = _sp.run(["find", "/root", "/content", "-name", "cloudflared.log"],
                         capture_output=True, text=True, timeout=5).stdout.strip()
        print(f"[ntfy] Could not find hostname. Log files: {_found or 'none'}")
except Exception as _e:
    print(f"[ntfy] Error: {_e!r}")

In [ ]:
# Mount Google Drive for persistent storage (runs after SSH so hostname is available first).
# Transient timeouts are common — see https://research.google.com/colaboratory/faq.html#drive-timeout
import time
from google.colab import drive

def _mount_drive():
    last_err = None
    for attempt in range(1, 4):
        try:
            # Longer timeout when supported (newer Colab runtimes).
            try:
                drive.mount("/content/drive", force_remount=False, timeout_ms=300_000)
            except TypeError:
                drive.mount("/content/drive", force_remount=False)
            return True
        except (ValueError, OSError, RuntimeError) as e:
            last_err = e
            print(f"[drive] mount attempt {attempt}/3 failed: {e!r}")
            if attempt < 3:
                time.sleep(20)
    print("[drive] Giving up on Drive mount — next cell will use /content only.")
    print(f"[drive] Last error: {last_err!r}")
    return False

_MOUNT_OK = _mount_drive()

In [ ]:
import os

if os.path.isdir('/content/drive/MyDrive'):
    WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
else:
    WORK_DIR = '/content/recsys_playground'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'

if not os.path.exists(repo_dir):
    !git clone {repo_url}

%cd {repo_dir}
!git pull origin main
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn requests papermill